## Weather Impact on Dublin Public Transport

analysing how weather conditions affect bus and luas ridership in dublin

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

just the basic libraries i need - pandas for data, matplotlib/seaborn for charts, scipy for stats tests

In [ ]:
weather_daily = pd.read_csv('data/met_eireann_dublin_daily.csv', skiprows=24)
weather_daily['date'] = pd.to_datetime(weather_daily['date'], format='%d-%b-%Y')
weather_daily.head()

met eireann daily weather data from dublin airport. the csv has 24 lines of metadata at the top so i skip them

In [ ]:
weather_monthly = pd.read_csv('data/met_eireann_dublin_monthly.csv', skiprows=19)
weather_monthly.head()

monthly weather averages - same station, 19 header lines to skip this time

In [ ]:
bus = pd.read_csv('data/dublin_bus_monthly_passengers.csv')
bus = bus[bus['Month'] != 'All months']
bus['passengers'] = pd.to_numeric(bus['VALUE'], errors='coerce')
bus['year'] = bus['Year'].astype(int)
bus['month'] = bus['Month'].map({
    'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
    'July':7,'August':8,'September':9,'October':10,'November':11,'December':12
})
bus.head()

dublin bus monthly passenger data from CSO. i filter out the 'All months' summary rows and convert the month names to numbers so i can merge later

In [ ]:
luas = pd.read_csv('data/luas_passenger_numbers.csv')
luas = luas[luas['Month'] != 'All months']
luas['passengers'] = pd.to_numeric(luas['VALUE'], errors='coerce')
luas['year'] = luas['Year'].astype(int)
luas['month'] = luas['Month'].map({
    'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
    'July':7,'August':8,'September':9,'October':10,'November':11,'December':12
})
luas.head()

luas (tram) passenger numbers - red and green line combined. same cleanup as bus data

In [ ]:
luas_hourly = pd.read_csv('data/luas_hourly_passengers.csv')
luas_hourly.head()

hourly distribution of luas passengers - shows what percentage travel at each hour

In [ ]:
weekly = pd.read_csv('data/weekly_passenger_journeys.csv')
weekly.head()

weekly passenger journey counts across different transport modes from the NTA

In [ ]:
print('weather daily:', weather_daily.shape)
print('weather monthly:', weather_monthly.shape)
print('bus:', bus.shape)
print('luas:', luas.shape)
print('luas hourly:', luas_hourly.shape)
print('weekly:', weekly.shape)

quick check to see how much data we have in each dataset

In [ ]:
print('--- weather daily missing ---')
print(weather_daily[['rain','maxtp','mintp']].isnull().sum())
print()
print('--- bus missing ---')
print(bus['passengers'].isnull().sum())
print()
print('--- luas missing ---')
print(luas['passengers'].isnull().sum())

checking for missing values in the key columns. weather data has some gaps in older records but its not a big deal for our analysis since transport data only goes back to 2014

In [ ]:
# add some useful columns to daily weather
weather_daily['avg_temp'] = (weather_daily['maxtp'] + weather_daily['mintp']) / 2
weather_daily['is_rainy'] = weather_daily['rain'] > 0
weather_daily['year'] = weather_daily['date'].dt.year
weather_daily['month'] = weather_daily['date'].dt.month

# season mapping
def get_season(m):
    if m in [12, 1, 2]: return 'Winter'
    elif m in [3, 4, 5]: return 'Spring'
    elif m in [6, 7, 8]: return 'Summer'
    else: return 'Autumn'

weather_daily['season'] = weather_daily['month'].apply(get_season)
weather_daily.head()

i create average temperature from max and min, a boolean for rainy days, and a season column. these will be useful for grouping later

In [ ]:
weather_agg = weather_daily.groupby(['year','month']).agg(
    avg_rain=('rain','mean'),
    avg_temp=('avg_temp','mean'),
    rainy_days=('is_rainy','sum'),
    avg_wind=('wdsp','mean')
).reset_index()
weather_agg.head()

aggregate daily weather to monthly averages so i can merge with the monthly transport data

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

recent = weather_daily[weather_daily['year'] >= 2014]
monthly_temp = recent.groupby(recent['date'].dt.to_period('M'))['avg_temp'].mean()
monthly_rain = recent.groupby(recent['date'].dt.to_period('M'))['rain'].sum()

axes[0].plot(monthly_temp.index.astype(str), monthly_temp.values, color='tomato', linewidth=0.8)
axes[0].set_title('Monthly Avg Temperature (2014-2025)')
axes[0].tick_params(axis='x', rotation=90, labelsize=6)

axes[1].bar(monthly_rain.index.astype(str), monthly_rain.values, color='steelblue', width=1)
axes[1].set_title('Monthly Total Rainfall (2014-2025)')
axes[1].tick_params(axis='x', rotation=90, labelsize=6)

plt.tight_layout()
plt.show()

overview of dublin weather since 2014. temperature follows a clear seasonal cycle. rainfall varies a lot month to month but winter tends to be wetter

In [ ]:
bus_yearly = bus.groupby('year')['passengers'].sum()
luas_yearly = luas.groupby('year')['passengers'].sum()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(bus_yearly.index, bus_yearly.values / 1e6, marker='o', label='Dublin Bus')
ax.plot(luas_yearly.index, luas_yearly.values / 1e6, marker='s', label='Luas')
ax.set_xlabel('Year')
ax.set_ylabel('Passengers (millions)')
ax.set_title('Annual Passengers')
ax.legend()
plt.tight_layout()
plt.show()

you can clearly see the covid drop in 2020-2021. bus has way more passengers than luas overall but both follow the same trend

In [ ]:
bus_weather = bus.merge(weather_agg, on=['year','month'], how='inner')
print('merged bus+weather rows:', len(bus_weather))
bus_weather.head()

merging bus passenger counts with monthly weather averages. inner join so we only keep months where both datasets have data

In [ ]:
luas_total = luas.groupby(['year','month'])['passengers'].sum().reset_index()
luas_weather = luas_total.merge(weather_agg, on=['year','month'], how='inner')
print('merged luas+weather rows:', len(luas_weather))
luas_weather.head()

same merge for luas - i group red+green line first then merge with weather

In [ ]:
bus_clean = bus_weather[~bus_weather['year'].isin([2020, 2021])]
luas_clean = luas_weather[~luas_weather['year'].isin([2020, 2021])]
print('bus rows after removing covid years:', len(bus_clean))
print('luas rows after removing covid years:', len(luas_clean))

removing 2020 and 2021 because covid completely messed up the patterns. including them would hide the real weather effects

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(bus_clean['avg_rain'], bus_clean['passengers']/1e6, alpha=0.6, color='steelblue')
axes[0].set_xlabel('Avg Daily Rain (mm)')
axes[0].set_ylabel('Passengers (millions)')
axes[0].set_title('Bus: Rain vs Passengers')

axes[1].scatter(luas_clean['avg_rain'], luas_clean['passengers']/1e6, alpha=0.6, color='teal')
axes[1].set_xlabel('Avg Daily Rain (mm)')
axes[1].set_ylabel('Passengers (millions)')
axes[1].set_title('Luas: Rain vs Passengers')

plt.tight_layout()
plt.show()

scatter plots showing relationship between rainfall and passenger numbers. doesnt show a super strong pattern visually but lets check with statistics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(bus_clean['avg_temp'], bus_clean['passengers']/1e6, alpha=0.6, color='tomato')
axes[0].set_xlabel('Avg Temperature (C)')
axes[0].set_ylabel('Passengers (millions)')
axes[0].set_title('Bus: Temp vs Passengers')

axes[1].scatter(luas_clean['avg_temp'], luas_clean['passengers']/1e6, alpha=0.6, color='coral')
axes[1].set_xlabel('Avg Temperature (C)')
axes[1].set_ylabel('Passengers (millions)')
axes[1].set_title('Luas: Temp vs Passengers')

plt.tight_layout()
plt.show()

temperature scatter plots. there seems to be a slight pattern here - lets quantify it

In [ ]:
# pearson and spearman correlations
for name, df in [('Bus', bus_clean), ('Luas', luas_clean)]:
    r_rain, p_rain = stats.pearsonr(df['avg_rain'], df['passengers'])
    r_temp, p_temp = stats.pearsonr(df['avg_temp'], df['passengers'])
    rs_rain, ps_rain = stats.spearmanr(df['avg_rain'], df['passengers'])
    rs_temp, ps_temp = stats.spearmanr(df['avg_temp'], df['passengers'])
    print(f'--- {name} ---')
    print(f'Rain  - Pearson r={r_rain:.3f} p={p_rain:.4f} | Spearman r={rs_rain:.3f} p={ps_rain:.4f}')
    print(f'Temp  - Pearson r={r_temp:.3f} p={p_temp:.4f} | Spearman r={rs_temp:.3f} p={ps_temp:.4f}')
    print()

correlation coefficients tell us how strong the relationship is. pearson measures linear relationship, spearman measures any monotonic relationship. p-value under 0.05 means its statistically significant

In [ ]:
cols = ['avg_rain', 'avg_temp', 'rainy_days', 'avg_wind', 'passengers']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(bus_clean[cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[0], vmin=-1, vmax=1)
axes[0].set_title('Bus - Correlation Matrix')

sns.heatmap(luas_clean[cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1], vmin=-1, vmax=1)
axes[1].set_title('Luas - Correlation Matrix')

plt.tight_layout()
plt.show()

heatmaps make it easier to see all the correlations at once. red means positive correlation, blue means negative. the diagonal is always 1 because each variable correlates perfectly with itself

In [ ]:
# split into rain categories
bus_clean = bus_clean.copy()
bus_clean['rain_cat'] = pd.cut(bus_clean['avg_rain'], bins=3, labels=['Low','Medium','High'])

groups = [g['passengers'].values for _, g in bus_clean.groupby('rain_cat') if len(g) > 0]
f_stat, p_val = stats.f_oneway(*groups)
print(f'ANOVA (bus passengers by rain level): F={f_stat:.3f}, p={p_val:.4f}')

# also kruskal-wallis (non-parametric)
h_stat, p_kw = stats.kruskal(*groups)
print(f'Kruskal-Wallis: H={h_stat:.3f}, p={p_kw:.4f}')

ANOVA tests if the mean passenger count is different across rain groups. kruskal-wallis is the non-parametric version that doesnt assume normal distribution. if p < 0.05 then rain level significantly affects ridership

In [ ]:
bus_clean['temp_cat'] = pd.cut(bus_clean['avg_temp'], bins=3, labels=['Cold','Mild','Warm'])

groups_t = [g['passengers'].values for _, g in bus_clean.groupby('temp_cat') if len(g) > 0]
f_stat_t, p_val_t = stats.f_oneway(*groups_t)
print(f'ANOVA (bus passengers by temp level): F={f_stat_t:.3f}, p={p_val_t:.4f}')

h_stat_t, p_kw_t = stats.kruskal(*groups_t)
print(f'Kruskal-Wallis: H={h_stat_t:.3f}, p={p_kw_t:.4f}')

same test but for temperature groups. this tells us if temperature has a significant effect on how many people take the bus

In [ ]:
# add season to merged data
def get_season(m):
    if m in [12,1,2]: return 'Winter'
    elif m in [3,4,5]: return 'Spring'
    elif m in [6,7,8]: return 'Summer'
    else: return 'Autumn'

bus_clean['season'] = bus_clean['month'].apply(get_season)
seasonal = bus_clean.groupby('season')['passengers'].mean().reindex(['Spring','Summer','Autumn','Winter'])

plt.figure(figsize=(8, 5))
seasonal.plot(kind='bar', color=['green','gold','orange','skyblue'])
plt.title('Average Monthly Bus Passengers by Season')
plt.ylabel('Passengers')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

seasonal breakdown shows autumn tends to have the most passengers (back to school/work) while summer dips a bit (holidays)

In [ ]:
hourly = luas_hourly[luas_hourly['Luas Line'] == 'All Luas lines'].copy()
hourly['pct'] = pd.to_numeric(hourly['VALUE'], errors='coerce')
hourly_avg = hourly.groupby('Time of day')['pct'].mean().sort_index()

plt.figure(figsize=(10, 5))
hourly_avg.plot(kind='bar', color='teal')
plt.title('Luas Passenger Distribution by Hour')
plt.ylabel('% of Daily Passengers')
plt.xlabel('Time')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

classic commuter pattern with morning peak around 8-9am and evening peak around 5-6pm. this is useful context for understanding when weather might matter most

In [ ]:
bus_all_yearly = bus.groupby('year')['passengers'].sum()

plt.figure(figsize=(8, 5))
colors = ['grey' if y in [2020,2021] else 'steelblue' for y in bus_all_yearly.index]
plt.bar(bus_all_yearly.index, bus_all_yearly.values / 1e6, color=colors)
plt.title('Dublin Bus Annual Passengers')
plt.ylabel('Passengers (millions)')
plt.xlabel('Year')
for i, (y, v) in enumerate(bus_all_yearly.items()):
    if y in [2020, 2021]:
        plt.annotate('COVID', (y, v/1e6), ha='center', va='bottom', fontsize=8, color='red')
plt.tight_layout()
plt.show()

the grey bars are covid years. passenger numbers dropped massively in 2020 and are still recovering. this is why i exclude them from the weather correlation analysis